The ASAP dataset contains paired MIDI score files (what *should* be played) and MIDI performance files (what a professional pianist *actually* played). We use the score as the reference and the performance as the student response, and run the ED alignment algorithm on each pair.

citation:

@inproceedings{asap-dataset,
  title={{ASAP}: a dataset of aligned scores and performances for piano transcription},
  author={Foscarin, Francesco and McLeod, Andrew and Rigaux, Philippe and Jacquemard, Florent and Sakai, Masahiko},
  booktitle={International Society for Music Information Retrieval Conference {(ISMIR)}},
  year={2020},
  pages={534--541}
}

# Download the ASAP Dataset

In [1]:
import os

if not os.path.exists('asap-dataset'):
    os.system('git clone https://github.com/fosfrancesco/asap-dataset.git')
else:
    print('asap-dataset already exists, skipping download.')

ASAP_PATH = 'asap-dataset'

Cloning into 'asap-dataset'...
Updating files: 100% (2854/2854), done.


## Format conversion

These functions convert a MIDI file into the `{pitch, start, duration}` format used by `compare_MIDI.py`.

In [3]:
import pretty_midi

def midi_file_to_notes(midi_path):
    """
    Parse a MIDI file and return all its notes in compareMusic format.
    Notes are sorted by onset time.
    """
    midi_data = pretty_midi.PrettyMIDI(midi_path)
    all_notes = []
    for instrument in midi_data.instruments:
        if instrument.is_drum: # Ignore drum tracks
            continue
        for note in instrument.notes:
            all_notes.append({
                "pitch":    note.pitch,
                "start":    round(note.start, 3),
                "duration": round(note.end - note.start, 3),
            })
    all_notes.sort(key=lambda n: (n["start"], n["pitch"]))
    return all_notes


def build_sample(ref_path, response_path, composer, title):
    """
    Convert one score/performance MIDI pair into a sample dict
    ready for compare_performance_ED.
    Returns None if either MIDI file produces zero notes.
    """
    score_notes = midi_file_to_notes(ref_path)
    perf_notes  = midi_file_to_notes(response_path)

    if not score_notes or not perf_notes:
        return None

    return {
        "composer":  composer,
        "title":     title,
        "reference": {"notes": score_notes},
        "response":  {"notes": perf_notes},
    }

# Load Score/Performance Pairs from ASAP Metadata

For the ASAP dataset, the `metadata.csv` lists every score/performance pair in the dataset, check that both MIDI files exist on disk.

In [6]:
import csv

def load_samples(asap_path, composer):
    """
    Read the ASAP metadata CSV and return a list of sample dicts
    for a specific composer only.
    """
    metadata_path = os.path.join(asap_path, "metadata.csv")
    samples = []

    with open(metadata_path, "r", encoding="utf-8") as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            if row.get("composer", "").strip() != composer:
                continue

            ref_path      = os.path.join(asap_path, row.get("midi_score", "").strip())
            response_path = os.path.join(asap_path, row.get("midi_performance", "").strip())

            if os.path.isfile(ref_path) and os.path.isfile(response_path):
                sample = build_sample(
                    ref_path, response_path,
                    row.get("composer", "Unknown"),
                    row.get("title",    "Unknown"),
                )
                if sample is not None:
                    samples.append(sample)

    return samples

samples = load_samples(ASAP_PATH, "Bach")

print("Loaded", len(samples), "score/performance pairs.")
print()
for i, sample in enumerate(samples):
    print(
        str(i + 1) + ".",
        sample["composer"], "/", sample["title"],
        " | score notes:",       len(sample["reference"]["notes"]),
        " | performance notes:", len(sample["response"]["notes"])
    )

Loaded 169 score/performance pairs.

1. Bach / Fugue_bwv_846  | score notes: 755  | performance notes: 754
2. Bach / Fugue_bwv_848  | score notes: 1429  | performance notes: 1435
3. Bach / Fugue_bwv_848  | score notes: 1429  | performance notes: 1438
4. Bach / Fugue_bwv_848  | score notes: 1429  | performance notes: 1429
5. Bach / Fugue_bwv_848  | score notes: 1429  | performance notes: 1431
6. Bach / Fugue_bwv_848  | score notes: 1429  | performance notes: 1433
7. Bach / Fugue_bwv_848  | score notes: 1429  | performance notes: 1438
8. Bach / Fugue_bwv_848  | score notes: 1429  | performance notes: 1440
9. Bach / Fugue_bwv_848  | score notes: 1429  | performance notes: 1431
10. Bach / Fugue_bwv_848  | score notes: 1429  | performance notes: 1430
11. Bach / Fugue_bwv_854  | score notes: 732  | performance notes: 739
12. Bach / Fugue_bwv_854  | score notes: 732  | performance notes: 738
13. Bach / Fugue_bwv_854  | score notes: 732  | performance notes: 734
14. Bach / Fugue_bwv_854  | sco

# Run Alignment on Each Pair

Passing each score/performance pair through `compare_performance_ED`, where score MIDI is the **reference** and the performance MIDI is the **response**.

In [7]:
from evaluation_function.compare_MIDI import compare_performance_ED

def run_alignment_on_samples(samples):
    """
    Run compare_performance_ED on every sample and return the results.

    Args:
        samples: list of sample dicts from load_samples().

    Returns:
        List of result dicts, each with keys:
            composer, title, stats, event_details, is_correct
    """
    results = []
    for sample in samples:
        result = compare_performance_ED(
            sample["response"],
            sample["reference"]
        )
        results.append({
            "composer":      sample["composer"],
            "title":         sample["title"],
            "stats":         result.stats,
            "event_details": result.event_details,
            "is_correct":    result.is_correct,
        })
    return results

all_results = run_alignment_on_samples(samples)
print("Alignment complete for", len(all_results), "pieces.")

Alignment complete for 169 pieces.


summary

In [9]:
import pandas as pd

def make_results_table(all_results):
    """Build a summary DataFrame from alignment results."""
    rows = []
    for r in all_results:
        stats = r["stats"]
        total = stats["total_notes_in_reference"]
        missing = stats["total_notes_missing"]
        wrong = stats["total_notes_wrong_pitch"]
        matched = total - missing - wrong
        rows.append({
            "Piece": r["composer"] + " / " + r["title"],
            "Score": total,
            "Missing": missing,
            "Extra": stats["total_notes_extra"],
            "Wrong": wrong,
            "Match %": round(100.0 * matched / total, 1) if total > 0 else 0.0,
        })
    return pd.DataFrame(rows)

make_results_table(all_results)

,Piece,Score,Missing,Extra,Wrong,Match %
0,Bach / Fugue_bwv_846,187,0,12,3,98.4
1,Bach / Fugue_bwv_848,313,4,4,7,96.5
2,Bach / Fugue_bwv_848,313,4,12,8,96.2
3,Bach / Fugue_bwv_848,313,2,11,14,94.9
4,Bach / Fugue_bwv_848,313,4,12,9,95.8
...,...,...,...,...,...,...
164,Bach / Prelude_bwv_892,440,16,7,7,94.8
165,Bach / Prelude_bwv_892,440,9,5,18,93.9
166,Bach / Prelude_bwv_893,428,7,5,2,97.9
167,Bach / Prelude_bwv_893,428,3,2,3,98.6


Timing Deviation Distribution

In [10]:
def collect_timing_deviations(all_results):
    """
    Collect local timing deviations from all matched notes across all pieces.

    Args:
        all_results: list of result dicts from run_alignment_on_samples().

    Returns:
        List of floats (one per matched note), in seconds.
    """
    deviations = []
    for result in all_results:
        for event in result["event_details"]:
            if event["event_type"] != "note":
                continue
            if event["operation_type"] != "match":
                continue
            if event.get("timing_deviation") is not None:
                deviations.append(event["timing_deviation"])
    return deviations


def print_deviation_summary(deviations, label):
    """
    Print a text summary of any list of deviation values.

    Args:
        deviations: list of floats.
        label: str, printed in the header line.
    """
    if not deviations:
        print("No", label, "deviations found.")
        return

    sorted_devs = sorted(deviations)
    n = len(sorted_devs)
    mean_abs = round(sum(abs(d) for d in sorted_devs) / n, 4)
    median = round(sorted_devs[n // 2], 4)
    pct_90 = round(sorted_devs[int(n * 0.90)], 4)
    pct_95 = round(sorted_devs[int(n * 0.95)], 4)

    within_50ms = sum(1 for d in sorted_devs if abs(d) <= 0.05)
    within_100ms = sum(1 for d in sorted_devs if abs(d) <= 0.10)

    print(label, "deviation summary across", n, "matched notes:")
    print("  Mean absolute : ", mean_abs, "s")
    print("  Median : ", median, "s")
    print("  90th pct : ", pct_90, "s")
    print("  95th pct : ", pct_95, "s")
    print("  Within 50 ms : ", round(100.0 * within_50ms / n, 1), "%")
    print("  Within 100 ms : ", round(100.0 * within_100ms / n, 1), "%")


timing_deviations = collect_timing_deviations(all_results)
print_deviation_summary(timing_deviations, "Timing")

No Timing deviations found.


Duration Deviation Distribution

In [11]:
def collect_duration_deviations(all_results):
    """
    Collect local duration deviations from all matched notes across all pieces.

    Args:
        all_results: list of result dicts from run_alignment_on_samples().

    Returns:
        List of floats (one per matched note), in seconds.
    """
    deviations = []
    for result in all_results:
        for event in result["event_details"]:
            if event["event_type"] != "note":
                continue
            if event["operation_type"] != "match":
                continue
            if event.get("duration_deviation") is not None:
                deviations.append(event["duration_deviation"])
    return deviations


duration_deviations = collect_duration_deviations(all_results)
print_deviation_summary(duration_deviations, "Duration")

No Duration deviations found.
